In [1]:
import os
import mne 
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
from pathlib import Path
import re

Cargar datos y corregir errores

In [22]:


def lerr_corregir_datos(rutare,rutara):

    df_records=pd.read_csv(rutare)
    excluir=df_records[df_records["Has more data"]==0]["Filename"].values


    df_ratings = pd.read_csv(rutara)
    df_ratings.loc[df_ratings["Filename"] == 'case123_s27.edf', "Filename"] = 'case132_s27.edf'
    df_ratings=df_ratings.drop(df_ratings[df_ratings["Filename"]=='case133_s27.edf'].index)
    df_ratings=df_ratings[~df_ratings["SR_NA1"].isna()]  
    return excluir, df_ratings 

 

Funciones para extraer las señales del edf, dividirlas en epocas y almacenarlas en un dataframe

In [13]:



def cargar_registro(path,channels=None,ep=True ):

    raw = mne.io.read_raw_edf(path, preload=True, verbose=False)   
    if channels != None:
       raw.pick(channels)
    raw.filter(0.3, 49, verbose=False)
    if ep==True:
    
        epochs = mne.make_fixed_length_epochs(
            raw,
            duration=10,   
            overlap=0,          
            preload=True,
            verbose=False
        )

        data = epochs.get_data() * 1e6 #microVolts (porque mne trabaja en V) 

    else:
        data = raw.get_data() * 1e6  

 
    return data



def crear_dataframe(ruta,channels=None, excluir=None,ep=True ):
    if excluir is None:
        excluir=[]
    df=pd.DataFrame(columns=["case","subject" ,"data"])
    ruta=Path(ruta)
    for archivo in ruta.glob("*.edf"):  
        if archivo.name not in excluir and "s13" not in archivo.name:#Esta linea es para filtrar los datos que no tienen un registro de emociones
            patron = r"case(\d+)_s(\d+)"
            match = re.search(patron, archivo.stem)
            
            if match:
                case_id = int(match.group(1))
                subject_id = int(match.group(2))
                
                data = cargar_registro(archivo,channels,ep )
                X=[case_id,subject_id,data]
                df.loc[len(df)] = X
             
    return df

Channel names: 
['Fp1', 'Fp2', 'AF7', 'AF3', 'AF4', 'AF8', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'Oz', 'O2', 'EOG-HL', 'EOG-HR', 'EOG-VU', 'EOG-VD', 'EMG']

In [3]:
#a=cargar_registro("REM_Turku/Data/PSG/case01_s10.edf")
 

Calcular potencia alfa en canales f3 y f4:

In [73]:


def bandpower(signal, sf, band, n_fft=1024):
    """
    signal: 1D array del canal (n_muestras)
    sf: frecuencia de muestreo
    band: (fmin, fmax)
    """
    fmin, fmax = band
    
    psd, freqs = mne.time_frequency.psd_array_welch(
        signal,
        sfreq=sf,
        fmin=fmin,
        fmax=fmax,
        n_fft=n_fft,
        n_overlap=n_fft//2,
        verbose=False
    )
     
    power = np.trapezoid(psd, freqs)
    return power



def get_features(faa):    
    feat_df=pd.DataFrame(columns=["case","faa_mean","faa_std","faa_p75","faa_min","faa_max","faa_median"])

    for i in faa.keys():
        feat_df.loc[len(feat_df)]=[int(i), np.mean(faa[i]),np.std(faa[i]),np.percentile(faa[i], 75), np.min(faa[i]),np.max(faa[i]),np.median(faa[i])]
    
    feat_df=feat_df.sort_values("case")
    return feat_df



def get_faa(df,band,sf,ep=True):
    bands = {
        "delta": (0.5, 4),
        "theta": (4, 7),
        "alpha": (8, 12),
        "beta":  (13, 30),
        "gamma": (30, 45)
    }
 


    f3_dict = {}
    f4_dict = {}
    if ep==True:
        for i, row in df.iterrows():
            f3_dict[str(row["case"])] = []
            f4_dict[str(row["case"])] = [] 

            for epoch in row["data"]:
                F3_alpha = bandpower(epoch[0], sf, bands[band])
                F4_alpha = bandpower(epoch[1], sf, bands[band])
                f3_dict[str(row["case"])].append(F3_alpha)
                f4_dict[str(row["case"])].append(F4_alpha)
        
        FAAA= {}
        for clave in f3_dict:
            if clave in f4_dict: 
                FAAA[clave] = np.log(np.array(f4_dict[clave])) - np.log(np.array(f3_dict[clave]))
        
        return get_features(FAAA)

    else:
        
        for i, row in df.iterrows():
            f3_dict[str(row["case"])] = []
            f4_dict[str(row["case"])] = [] 
    
            F3_alpha = bandpower(row["data"][0], sf, bands[band])
            F4_alpha = bandpower(row["data"][1], sf, bands[band])
            f3_dict[str(row["case"])].append(F3_alpha)
            f4_dict[str(row["case"])].append(F4_alpha)



        FAAA= pd.DataFrame(columns=["case","faa"])
        for clave in f3_dict:
            if clave in f4_dict: 
                

                FAAA.loc[len(FAAA)]=[int(clave),np.log(np.array(f4_dict[clave])) - np.log(np.array(f3_dict[clave]))] 
              
            
        
        return FAAA.sort_values("case")



In [ ]:
#Version dividiendo en epocas
ruta_edf =  "REM_Turku/Data/PSG"
ruta_records= "REM_Turku/Records.csv"
ruta_ratings="REM_Turku/Data/Ratings.csv"

excluir, df_ratings=lerr_corregir_datos(ruta_records,ruta_ratings)

preFAA=crear_dataframe(ruta_edf,["F3","F4"],excluir)

faa=get_faa(preFAA,"alpha",500)


 

In [74]:
#Version sin dividir en epocas
excluir, df_ratings=lerr_corregir_datos(ruta_records,ruta_ratings)

preFAA_1=crear_dataframe(ruta_edf,["F3","F4"],excluir, ep=False)

faa_1=get_faa(preFAA_1,"alpha",500, ep=False)

Estudiar correlaciones

In [76]:
def get_corr(faa,filter,df_emotions,method="pearson"):
    df_corr = pd.concat([faa, df_emotions], axis=1).corr(method=method)
    df_corr= df_corr[df_corr.abs() > filter] 
    return df_corr


In [77]:
emotions=[col for col in df_ratings.columns if col.startswith("SR_")]
df_emotions=df_ratings[emotions]

 
corr_epoch_p=get_corr(faa,0.1,df_emotions)
corr_epoch_s=get_corr(faa,0.1,df_emotions,"spearman")
 
corr_no_epoch_p=get_corr(faa_1,0.1,df_emotions)
corr_no_epoch_s=get_corr(faa_1,0.1,df_emotions,"spearman")

En general la correlacion es casi nula, por lo que no espero obtener resultados con estos modelos. Voy a usar como target la emocion desden/desprecio(N3) ya que se trata de la que tiene una mayor correlacion, sin llegar a ser destacable

In [87]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

X=faa_1[["faa"]]
y=df_emotions["SR_NA3"]

X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.2)
modelo=LinearRegression()

modelo.fit(X_train,y_train)
y_pred = modelo.predict(X_test)

# Evaluar el modelo
print("Coeficientes:", modelo.coef_)
print("Intercepto:", modelo.intercept_)
print("R² Score:", r2_score(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))


: 

Supuesto:

FAA positivo->emociones reactivas(aproach-related)

FAA negativo-> emociones estaticas(Withdrawal-related)

Hecho:

En estos datos se ha encontrado correlacion entre el enfado(reactiva) y FAA

In [86]:
emotions=[col for col in df_ratings.columns if col.startswith("SR_")] 
df_ratings=df_ratings[df_ratings["SR_NA1"].isna()==False] 
df_emotions=df_ratings[emotions]

In [87]:
lista1=[]


for col in X.columns:
    for emotion in df_emotions:
        #print("aaaaaaaaaaaaaaaaaaa", emotion)
        corr = np.corrcoef(X[col], df_emotions[emotion])[0,1]
        #print(col, corr)
        lista1.append(corr)

In [89]:
np.max(lista1)

np.float64(0.20004535985112737)

In [47]:
sumas = {clave: sum(valores) for clave, valores in faa.items()}
print(sumas)

{'2': np.float64(-2.268186396148987), '3': np.float64(-1.3347543904774364), '4': np.float64(-1.6886578860729582), '5': np.float64(-1.1796320263354318), '7': np.float64(-2.596525075625312), '8': np.float64(-2.1951132364024053), '9': np.float64(-3.687531952057781), '100': np.float64(2.410195722765457), '101': np.float64(0.2777217484208634), '102': np.float64(1.715524562094196), '103': np.float64(0.4594761813528856), '104': np.float64(0.6053667292262812), '105': np.float64(2.090070807039512), '106': np.float64(0.7432966616663963), '107': np.float64(1.1723133750340466), '108': np.float64(-0.8786529386461865), '109': np.float64(-0.3259755766637067), '10': np.float64(-3.3464132981424473), '111': np.float64(-0.16454712376104763), '112': np.float64(1.1723133750340466), '113': np.float64(-0.7359678150334421), '114': np.float64(-2.702597809752339), '115': np.float64(-2.4122138793285393), '116': np.float64(-2.736175231766829), '117': np.float64(-1.9425465263233572), '118': np.float64(-2.669119177

Posible ground truth:

In [ ]:
def clasificar_emociones(row):


    
    if row["pa_sum"] > 2 * row["na_sum"] and row["pa_sum"] > 5:
        return "POSITIVO" 
    else:
        return "NO POSITIVO"#Las emociones negativas se reportan mucho menos, en esta categoria la mayoria de sueños son "neutrales", es decir, los indices de emociones son muy bajos para considerarse positivos o negativos.
        #De las 53 filas que entran en no positivo, solo cuatro pasarian un filtro de este tipo: row["pa_sum"] > 2 * row["na_sum"] and row["pa_sum"] > 5 pero para emociones negativas

In [ ]:
 
df = pd.read_csv("REM_Turku/Data/Ratings.csv")


df=df[df["Filename"]!="case133_s27.edf"]#Este cambio es porque falta un edf en los datos
df.loc[df["Filename"] == 'case123_s27.edf', "Filename"] = 'case132_s27.edf'#Este cambio es por un typo en el csv
pa=[col for col in df.columns if col.startswith("SR_PA")] 
na=[col for col in df.columns if col.startswith("SR_NA")]  
df["pa_sum"] = df[pa].sum(axis=1)
df["na_sum"] = df[na].sum(axis=1)    

df['sentimiento'] = df.apply(clasificar_emociones, axis=1)



SyntaxError: invalid syntax (227527097.py, line 7)